# 02 – Data Engineering (Aggregation & Feature-Matrix)

**CRISP-DM-Phase:** Data Preparation.
Ziel: aus dem **relationalen** Schema einen **flachen** Modellierungsdatensatz auf
Ebene `SK_ID_CURR` (eine Zeile pro Antragsteller) erzeugen.

**Kernproblem:** Nebentabellen stehen in 1:n-Beziehung zur Haupttabelle. Ein naiver
Join würde Antragszeilen vervielfachen ("row explosion"). Lösung: **erst aggregieren,
dann left-joinen**, und die Zeilenzahl als Invariante prüfen.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from src import config, load_data, data_cleaning, feature_engineering as fe, utils
log = utils.get_logger()
pd.set_option("display.max_columns", 120)

## 1. Daten laden & Haupttabelle bereinigen

**Was:** Pflicht- (und optionale) Tabellen laden, Haupttabelle deterministisch
bereinigen (Sentinel-Fix, Duplikate, unmögliche Werte).
**Warum:** Cleaning *vor* der Aggregation, damit z. B. der `DAYS_EMPLOYED`-Sentinel
nicht in Aggregate einfließt. Es werden **nur leakage-freie** Korrekturen angewandt
(keine geschätzten Statistiken).

In [2]:
tables = load_data.load_all_available()
tables["application_train"] = data_cleaning.clean_application(tables["application_train"])
data_cleaning.diagnose(tables["application_train"], "application_train (clean)").head(8)

[load_data] lade application_train ...
           shape=(307511, 122)
[load_data] lade application_test ...
           shape=(48744, 121)
[load_data] lade bureau ...
           shape=(1716428, 17)
[load_data] lade bureau_balance ...
           shape=(27299925, 3)
[load_data] lade previous_application ...
           shape=(1670214, 37)
[load_data] lade installments_payments ...
           shape=(13605401, 8)
[load_data] lade POS_CASH_balance ...
           shape=(10001358, 8)
[load_data] lade credit_card_balance ...


17:17:23 | INFO    | ID 'SK_ID_CURR' ist eindeutig (307511 Zeilen).
17:17:23 | INFO    | DAYS_EMPLOYED-Sentinel in 55374 Zeilen -> NaN + Indikator.
17:17:23 | INFO    | CODE_GENDER 'XNA' in 4 Zeilen -> NaN.


           shape=(3840312, 23)


17:17:24 | INFO    | Diagnose 'application_train (clean)': 307511 Zeilen x 123 Spalten


,Spalte,Dtype,Fehlend_%,Eindeutige_Werte,Top_Wert_Anteil_%
0,COMMONAREA_AVG,float32,69.87,3181,2.75
1,COMMONAREA_MEDI,float32,69.87,3202,2.83
2,COMMONAREA_MODE,float32,69.87,3128,3.15
3,NONLIVINGAPARTMENTS_MODE,float32,69.43,167,19.27
4,NONLIVINGAPARTMENTS_MEDI,float32,69.43,214,18.24
5,NONLIVINGAPARTMENTS_AVG,float32,69.43,386,17.74
6,FONDKAPREMONT_MODE,str,68.39,4,24.01
7,LIVINGAPARTMENTS_MEDI,float32,68.35,1097,1.46


## 2. Aggregation der Nebentabellen

Wir betrachten exemplarisch die `bureau`-Aggregation. Jede Kennzahl hat eine fachliche
Bedeutung (z. B. Anzahl überfälliger Fremdkredite als Verzugsindikator).
**Limitation:** Aggregate verdichten Information und verlieren zeitliche Dynamik
(z. B. *wann* ein Verzug auftrat). Das ist ein bewusster Trade-off zugunsten eines
tabellarischen Modells.

In [3]:
bureau_agg = fe.aggregate_bureau(tables["bureau"], tables.get("bureau_balance"))
print("bureau_agg:", bureau_agg.shape)
bureau_agg.head()

17:17:27 | INFO    | bureau aggregiert -> 305811 Antragsteller, 12 Features.


bureau_agg: (305811, 13)


,SK_ID_CURR,BUREAU_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_OVERDUE_COUNT,BUREAU_MEAN_AMT_CREDIT_SUM,BUREAU_SUM_AMT_CREDIT_SUM,BUREAU_MAX_AMT_CREDIT_SUM,BUREAU_MEAN_DAYS_CREDIT,BUREAU_MIN_DAYS_CREDIT,BUREAU_CREDIT_TYPE_NUNIQUE,BUREAU_BB_OVERDUE_MONTHS_SUM,BUREAU_BB_MONTHS_COUNT_SUM
0,100001,7,3,4,0,207623.571429,1453365.000,378000.0,-735.000000,-1572,1,1.0,172.0
1,100002,8,2,6,0,108131.945625,865055.565,450000.0,-874.000000,-1437,2,27.0,110.0
2,100003,4,1,3,0,254350.125000,1017400.500,810000.0,-1400.750000,-2586,2,0.0,0.0
3,100004,2,0,2,0,94518.900000,189037.800,94537.8,-867.000000,-1326,1,0.0,0.0
4,100005,3,2,1,0,219042.000000,657126.000,568800.0,-190.666667,-373,2,0.0,21.0


In [4]:
prev_agg = fe.aggregate_previous(tables["previous_application"])
inst_agg = fe.aggregate_installments(tables["installments_payments"])
print("prev_agg:", prev_agg.shape, "| inst_agg:", inst_agg.shape)
inst_agg.describe().T

17:17:28 | INFO    | previous_application aggregiert -> 338857 Antragsteller, 7 Features.
17:17:29 | INFO    | installments_payments aggregiert -> 339587 Antragsteller, 7 Features.


prev_agg: (338857, 8) | inst_agg: (339587, 8)


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,339587.0,278154.892278,102880.492598,100001.000,189042.500000,278238.000000,367315.500000,4.562550e+05
INST_COUNT,339587.0,40.064552,41.053343,1.000,12.000000,25.000000,51.000000,3.720000e+02
INST_PAYMENT_DELAY_MEAN,339578.0,-11.257627,12.976502,-295.000,-14.846154,-9.564102,-5.888889,1.884205e+03
INST_PAYMENT_DELAY_MAX,339578.0,15.695113,108.718147,-156.000,-2.000000,1.000000,9.000000,2.884000e+03
INST_LATE_PAYMENT_COUNT,339587.0,3.376658,6.374030,0.000,0.000000,1.000000,4.000000,1.590000e+02
INST_PAYMENT_DIFF_MEAN,339578.0,-390.786264,5242.747778,-337496.805,0.000000,0.000000,435.855153,1.461459e+05
INST_PAYMENT_DIFF_SUM,339587.0,-7496.631003,177788.623639,-4417384.230,0.000000,0.000000,15704.617500,3.037736e+06
INST_LATE_PAYMENT_RATIO,339587.0,0.074385,0.114490,0.000,0.000000,0.017857,0.109375,1.000000e+00


## 3. Finale Feature-Matrix bauen

`build_feature_matrix` fügt application-Features hinzu, hängt die Aggregate per
**Left-Join** an und prüft per `assert`, dass die Zeilenzahl exakt der Zahl der
Antragsteller entspricht (Schutz gegen row explosion).

In [5]:
feat = fe.build_feature_matrix(tables)
print("Feature-Matrix:", feat.shape)
assert feat[config.ID_COLUMN].is_unique, "IDs nicht eindeutig!"
print("ID eindeutig:", feat[config.ID_COLUMN].is_unique)
feat.filter(regex="^(BUREAU_|PREV_|INST_)").describe().T.head(12)

17:17:30 | INFO    | application-Features ergänzt -> 135 Spalten.
17:17:30 | INFO    | Start build_feature_matrix mit 307511 Antragstellern.
17:17:32 | INFO    | bureau aggregiert -> 305811 Antragsteller, 12 Features.
17:17:33 | INFO    | previous_application aggregiert -> 338857 Antragsteller, 7 Features.
17:17:34 | INFO    | installments_payments aggregiert -> 339587 Antragsteller, 7 Features.
17:17:34 | INFO    | Finale Feature-Matrix: 307511 Zeilen x 161 Spalten.


Feature-Matrix: (307511, 161)
ID eindeutig: True


,count,mean,std,min,25%,50%,75%,max
BUREAU_COUNT,307511.0,4.765114e+00,4.496199e+00,0.0,1.00,4.000000,7.000000e+00,1.160000e+02
BUREAU_ACTIVE_COUNT,307511.0,1.762275e+00,1.804891e+00,0.0,0.00,1.000000,3.000000e+00,3.200000e+01
BUREAU_CLOSED_COUNT,307511.0,2.984391e+00,3.359529e+00,0.0,0.00,2.000000,4.000000e+00,1.080000e+02
BUREAU_OVERDUE_COUNT,307511.0,1.184998e-02,1.212224e-01,0.0,0.00,0.000000,0.000000e+00,8.000000e+00
BUREAU_MEAN_AMT_CREDIT_SUM,263490.0,3.780802e+05,8.916731e+05,0.0,103500.00,195507.184091,3.941162e+05,1.980723e+08
BUREAU_SUM_AMT_CREDIT_SUM,263491.0,1.955807e+06,4.101728e+06,0.0,343377.27,961704.000000,2.297721e+06,1.017958e+09
BUREAU_MAX_AMT_CREDIT_SUM,263490.0,9.760041e+05,2.087566e+06,0.0,180000.00,450000.000000,1.043914e+06,3.960000e+08
BUREAU_MEAN_DAYS_CREDIT,263491.0,-1.083047e+03,5.633273e+02,-2922.0,-1434.00,-1050.571429,-6.637639e+02,0.000000e+00
BUREAU_MIN_DAYS_CREDIT,263491.0,-1.762375e+03,8.638622e+02,-2922.0,-2583.00,-1827.000000,-1.035000e+03,0.000000e+00
BUREAU_CREDIT_TYPE_NUNIQUE,263491.0,1.743103e+00,6.407146e-01,1.0,1.00,2.000000,2.000000e+00,6.000000e+00


## 4. Leakage- & Ethik-Review

**Was:** Wir markieren sensible/Proxy-Merkmale und prüfen auf TARGET-abhängige
Features.
**Warum:** Transparenz. Diese Spalten werden nicht blind verwendet, sondern im
Modellierungs- und Ethikteil bewusst diskutiert (z. B. Geschlecht/Alter als rechtlich
heikle Merkmale; mögliche indirekte Diskriminierung über Proxys).

In [6]:
review = fe.potential_leakage_and_ethics_review(feat)
print("Sensible/Proxy-Merkmale:", review["sensitive_or_proxy"])
print("Leakage-Kandidaten:", review["leakage_candidates"])

17:17:35 | INFO    | Ethik/Leakage-Review: 7 sensible, 0 Leakage-Kandidaten.


Sensible/Proxy-Merkmale: ['CODE_GENDER', 'DAYS_BIRTH', 'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'NAME_HOUSING_TYPE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']
Leakage-Kandidaten: []


## 5. Persistenz (interim)

**Was:** Feature-Matrix als Parquet in `data/processed/` speichern.
**Warum:** Reproduzierbarkeit & Effizienz – nachfolgende Notebooks (EDA, Modellierung)
laden das fertige Artefakt, statt die Aggregation erneut zu rechnen. Parquet bewahrt
Datentypen und ist schnell.

In [7]:
path = utils.save_parquet(feat, "feature_matrix", stage="processed")
print("Gespeichert:", path)

17:17:37 | INFO    | Parquet gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\data\processed\feature_matrix.parquet ((307511, 161))


Gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\data\processed\feature_matrix.parquet


## 6. Fazit

- Aus 4 (+1 optionalen) relationalen Tabellen entstand **eine** flache Matrix auf
  Antragsteller-Ebene.
- Die **Row-Explosion-Invariante** wurde verifiziert.
- Sensible Merkmale sind dokumentiert.

**Nächstes Notebook (03 – EDA):** Verteilungen, Korrelationen, Vergleich Ausfall vs.
Nicht-Ausfall, erste Hypothesen.